# <font color="#2196F3">**Synthetic Clinical QA Generation for GraphRAG Evaluation**</font><br/>
### Generating Factual Evaluation Pairs from PMC-Patients Case Summaries

This notebook generates factual question-answer pairs from the clinical patient records in **`eval/data/pmc_cut.jsonl`**.

#### **Question Types Generated**
1. **Chief Complaint & Symptoms**: Presenting symptoms and initial vital signs.
2. **Diagnosis & Condition**: Primary clinical findings and confirmed conditions.
3. **Interventions & Therapies**: Medications, surgical procedures, and therapies.
4. **Outcomes & Discharge**: Clinical resolution, discharge destinations, or prognosis.

These QA pairs are saved to `eval/data/synthetic_qa_pairs.csv` and used to evaluate GraphRAG retrieval accuracy against ground truth clinical facts.

## 1️⃣ Import Required Dependencies & Setup Paths

In [1]:
import os
import sys
import json
import re
from pprint import pprint
import pandas as pd

# Workspace paths
NOTEBOOK_DIR = os.getcwd()
DATA_PREP_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in locals() else NOTEBOOK_DIR
WORKSPACE_ROOT = os.path.abspath(os.path.join(DATA_PREP_DIR, "..", "..", ".."))

CUT_JSONL = os.path.join(WORKSPACE_ROOT, "backend", "eval", "data", "pmc_cut.jsonl")
QA_OUTPUT_CSV = os.path.join(WORKSPACE_ROOT, "backend", "eval", "data", "synthetic_qa_pairs.csv")
LOCAL_QA_CSV = os.path.join(WORKSPACE_ROOT, "GTC25_DLI", "data", "synthetic_qa_pairs.csv")

print(f"Input Dataset:  {CUT_JSONL}")
print(f"QA Output CSV:  {QA_OUTPUT_CSV}")

Input Dataset:  /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/pmc_cut.jsonl
QA Output CSV:  /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/synthetic_qa_pairs.csv


## 2️⃣ Load Patient Case Summaries

In [2]:
def load_patient_records(cut_file=CUT_JSONL):
    """Load patient records from JSONL."""
    if not os.path.exists(cut_file):
        raise FileNotFoundError(f"Missing {cut_file}. Run 2_SEC_Data_Preparation.ipynb first.")
    with open(cut_file, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(records)} patient case records.")
    return records

patient_records = load_patient_records()

Loaded 100 patient case records.


## 3️⃣ Synthetic Clinical QA Pair Generation Logic
We generate factual, testable questions directly grounded in the patient text.

In [3]:
def generate_clinical_qa_pairs(records):
    """
    Generates structured factual clinical Q&A pairs for each patient case.
    Covers demographics, presentation, diagnosis, treatment, and outcomes.
    """
    qa_rows = []
    
    for r in records:
        pid = r["patient_id"]
        age = r["age"]
        gender = r["gender"]
        cond = r["condition_hint"]
        text = r["text"]
        
        # 1. Demographics & Presentation Question
        q1 = f"What was the initial clinical presentation and demographic profile of patient {pid}?"
        # Extract first two sentences for presentation summary
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
        a1 = " ".join(sentences[:2]) if len(sentences) >= 2 else sentences[0]
        qa_rows.append({
            "patient_id": pid,
            "category": "presentation",
            "question": q1,
            "ground_truth_answer": a1,
            "condition_hint": cond
        })
        
        # 2. Diagnosed Condition Question (if condition hint is present)
        if cond:
            q2 = f"What primary clinical condition or disease was identified for the {age} {gender} patient {pid}?"
            # Find the sentence mentioning the condition
            cond_sentence = next((s for s in sentences if cond.lower() in s.lower()), sentences[0])
            a2 = f"Patient {pid} presented with or was diagnosed with {cond.upper()}. Details: {cond_sentence}"
            qa_rows.append({
                "patient_id": pid,
                "category": "diagnosis",
                "question": q2,
                "ground_truth_answer": a2,
                "condition_hint": cond
            })
            
        # 3. Interventions / Therapy Question
        intervention_kw = ["therapy", "treated", "administered", "surgery", "oxygen", "medication", "dose", "mg", "rehabilitation"]
        rx_sentence = next((s for s in sentences if any(k in s.lower() for k in intervention_kw)), None)
        if rx_sentence:
            q3 = f"What clinical management, therapy, or interventions were performed for patient {pid}?"
            a3 = rx_sentence
            qa_rows.append({
                "patient_id": pid,
                "category": "intervention",
                "question": q3,
                "ground_truth_answer": a3,
                "condition_hint": cond
            })
            
        # 4. Discharge / Outcome Question
        outcome_kw = ["discharge", "recovered", "improved", "died", "death", "follow-up", "stable", "hospital"]
        outcome_sentence = next((s for s in reversed(sentences) if any(k in s.lower() for k in outcome_kw)), sentences[-1])
        q4 = f"What was the clinical outcome or discharge status for patient {pid}?"
        a4 = outcome_sentence
        qa_rows.append({
            "patient_id": pid,
            "category": "outcome",
            "question": q4,
            "ground_truth_answer": a4,
            "condition_hint": cond
        })
        
    df_qa = pd.DataFrame(qa_rows)
    print(f"Generated {len(df_qa)} synthetic clinical QA pairs across {len(records)} patients.")
    return df_qa

df_qa = generate_clinical_qa_pairs(patient_records)

Generated 326 synthetic clinical QA pairs across 100 patients.


## 4️⃣ Save QA Dataset to CSV and Inspect Samples

In [4]:
# Save generated QA pairs to evaluation directories
for path in [QA_OUTPUT_CSV, LOCAL_QA_CSV]:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df_qa.to_csv(path, index=False)
    print(f"✅ Saved synthetic QA dataset -> {path}")

# Display distribution by category
print("\n--- Distribution by Question Category ---")
print(df_qa["category"].value_counts())

# Inspect sample QA entries
print("\n--- Sample Question-Answer Pairs ---")
for _, row in df_qa.head(4).iterrows():
    print(f"[{row['category'].upper()}] Patient {row['patient_id']}")
    print(f"  Q: {row['question']}")
    print(f"  A: {row['ground_truth_answer'][:150]}...\n")

✅ Saved synthetic QA dataset -> /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/backend/eval/data/synthetic_qa_pairs.csv
✅ Saved synthetic QA dataset -> /Users/signatur4ik/Desktop/Determenistic_memory_layer/knowledge_graph_rag/GTC25_DLI/data/synthetic_qa_pairs.csv

--- Distribution by Question Category ---
category
presentation    100
outcome         100
intervention     78
diagnosis        48
Name: count, dtype: int64

--- Sample Question-Answer Pairs ---
[PRESENTATION] Patient 0
  Q: What was the initial clinical presentation and demographic profile of patient 0?
  A: This 60-year-old male was hospitalized due to moderate ARDS from COVID-19 with symptoms of fever, dry cough, and dyspnea. We encountered several diffi...

[DIAGNOSIS] Patient 0
  Q: What primary clinical condition or disease was identified for the 60 years male patient 0?
  A: Patient 0 presented with or was diagnosed with ARDS. Details: This 60-year-old male was hospitalized due to moderate AR

## 5️⃣ GraphRAG Evaluation Connection
Integrate with the deterministic knowledge graph backend (`dag_kb` / `service.assembly`) to evaluate retrieval quality:

In [5]:
def test_graphrag_retrieval():
    try:
        sys.path.insert(0, os.path.join(WORKSPACE_ROOT, 'backend'))
        from service.assembly import build_memory_layer
        
        data_dir = os.path.join(WORKSPACE_ROOT, 'backend', 'data')
        if not os.path.exists(os.path.join(data_dir, 'dag.graphml')):
            print('Note: Knowledge DAG graphml file not found yet. Run stage 1 extraction to build graph.')
            return
            
        layer = build_memory_layer(data_dir=data_dir)
        nodes_count = len(layer.dag._g.nodes)
        edges_count = len(layer.dag._g.edges)
        print(f'Connected to Knowledge DAG with {nodes_count} nodes and {edges_count} edges.')
    except Exception as e:
        print(f'GraphRAG retrieval test notice: {e}')

# Run connectivity check
test_graphrag_retrieval()


Connected to Knowledge DAG with 187 nodes and 212 edges.
